# Models on edge devices, GPU monitoring, Netron, TFLite/LiteRT and TensorFlow.js

The notebook presents different runtime environments for ML models: GPU, edge, IoT, and running in the browser.


In [1]:
import tensorflow as tf
from pathlib import Path

(x_train, y_train), _ = tf.keras.datasets.mnist.load_data()
x_train = x_train.astype("float32") / 255.0

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
model.fit(x_train[:5000], y_train[:5000], epochs=1, batch_size=128)

Path("models").mkdir(exist_ok=True)
model.save("models/mnist_small.keras")
print("Saved Keras model: models/mnist_small.keras")


I0000 00:00:1781900595.798574    1502 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1781900595.845196    1502 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI AVX_VNNI_INT8 AVX_NE_CONVERT FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1781900597.055592    1502 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
****************************************************************************
* hwloc 2.0.3 received inval

40/40 [==============================] - 22s 3ms/step - loss: 1.6340 - accuracy: 0.5334
Saved Keras model: models/mnist_small.keras


## TensorFlow Lite

In [2]:
# Convert to TensorFlow Lite / LiteRT
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("models/mnist_small.tflite", "wb") as f:
    f.write(tflite_model)

with open("models/mnist_small.keras", "rb") as f:
    model_bytes = f.read()

print("TF model size:", len(model_bytes) / 1024, "KB")
print("TFLite model size:", len(tflite_model) / 1024, "KB")


INFO:tensorflow:Assets written to: /tmp/tmptbhixp0i/assets


INFO:tensorflow:Assets written to: /tmp/tmptbhixp0i/assets


TF model size: 321.2060546875 KB
TFLite model size: 101.34375 KB


W0000 00:00:1781900622.164450    1502 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1781900622.164465    1502 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1781900622.164711    1502 reader.cc:83] Reading SavedModel from: /tmp/tmptbhixp0i
I0000 00:00:1781900622.165403    1502 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1781900622.165411    1502 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmptbhixp0i
I0000 00:00:1781900622.170224    1502 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1781900622.170867    1502 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1781900622.198103    1502 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmptbhixp0i
I0000 00:00:1781900622.204682    1502 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 39982 microseconds.


## Netron

Netron allows you to quickly view the model's architecture.

Setup:
1. Go to `https://netron.app` and drag the `models/mnist_small.keras` file.
2. Install locally:

```bash
pip install netron
netron models/mnist_small.keras
```

## nvidia-smi ? GPU monitoring

Basic commands to run locally:

```bash
nvidia-smi
nvidia-smi --loop=1
nvidia-smi dmon
nvidia-smi pmon
```

What resources can be monitored in nvidia-smi?
- GPU utilization;
- Memory usage;
- Processes using the GPU;
- Temperature and power consumption;
- Whether the model is actually using the GPU;
- Whether the bottleneck is in the GPU, CPU, disk, dataloader, or batch size.


In [3]:
# GPU check from TensorFlow
import tensorflow as tf
print(tf.config.list_physical_devices("GPU"))


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## TensorFlow.js

TensorFlow.js allows you to run models in a browser or Node.js.

Exporting can be done using the `tensorflowjs_converter` tool:

```bash
pip install tensorflowjs
tensorflowjs_converter --input_format=keras models/mnist_small.keras tfjs_model/
```

In the browser, the model can be loaded via `tf.loadLayersModel(...)`.


## Summary

- TFLite/LiteRT is used for edge and on-device inference: mobile, IoT, embedded, and industrial devices.
- TensorFlow.js is useful when the model needs to run directly in the browser.
- Netron helps you understand the model structure before deployment.
- `nvidia-smi` is the simplest tool to quickly check whether the model is actually using the GPU and how many resources it's consuming.
